# DFG Analysis: Loops, Short Loops, and Parallels

Build a Directly-Follows Graph (DFG) from an event log and detect:
- **Short loops** â€” self-loops (aâ†’a) and 2-node cycles (aâ†’b, bâ†’a)
- **Parallels** â€” bidirectional edges (aâ†’b and bâ†’a) that may indicate concurrency
- **Longer loops/cycles** â€” multi-node cycles traversing the DFG

## Setup
Import required libraries.

In [1]:
import pm4py
from collections import Counter, defaultdict
from pm4py.objects.conversion.log import converter as log_converter

## Load Event Log
Load the repair example event log (has enough behavioral diversity).

In [ ]:
LOG_PATH = "../../data/villach/processflow_export.xes"
event_log = log_converter.apply(pm4py.read_xes(LOG_PATH))

print(f"Traces: {len(event_log)}")

c:\Users\safaya\Documents\GitHub\process-fragment-miner\.venv\Lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/53 [00:00<?, ?it/s]

Traces: 53


## Extract Activity Sequences
Convert traces to activity sequences for DFG construction.

In [3]:
sequences = []
for trace in event_log:
    seq = [event["concept:name"] for event in trace]
    sequences.append(seq)

print(f"Sequences: {len(sequences)}")
print(f"First sequence: {sequences[0]}")

Sequences: 53
First sequence: ['1017', '1028', '1029', '1033', '1035', '2200', '2230', '2250', '9020', '9030', '9050', '9841', '9922', '9924', '9996']


## Build Directly-Follows Graph
Count how often each activity is directly followed by another. The result is a weighted directed graph.

In [4]:
dfg_counter = Counter()
for seq in sequences:
    for i in range(len(seq) - 1):
        pair = (seq[i], seq[i + 1])
        dfg_counter[pair] += 1

print(f"Total DFG edges: {len(dfg_counter)}")
print()
print("All DFG edges (sorted by frequency):")
for (a, b), w in dfg_counter.most_common():
    print(f"  {a:25s} â†’ {b:25s}  (x{w})")

Total DFG edges: 31

All DFG edges (sorted by frequency):
  2230                      â†’ 2250                       (x53)
  9020                      â†’ 9030                       (x53)
  9030                      â†’ 9050                       (x53)
  9841                      â†’ 9922                       (x53)
  9922                      â†’ 9924                       (x53)
  9924                      â†’ 9996                       (x53)
  9050                      â†’ 9841                       (x49)
  1017                      â†’ 1028                       (x37)
  1028                      â†’ 1029                       (x37)
  1029                      â†’ 1033                       (x37)
  1033                      â†’ 1035                       (x37)
  2200                      â†’ 2230                       (x31)
  2250                      â†’ 9020                       (x27)
  2250                      â†’ 2255                       (x19)
  2255                      â†’ 

## 1. Short Loops â€” Self-Loops (aâ†’a)
A self-loop occurs when the same activity appears consecutively in a trace.

In [5]:
self_loops = {(a, b): w for (a, b), w in dfg_counter.items() if a == b}

print(f"Self-loops found: {len(self_loops)}")
print()
for (a, _), w in sorted(self_loops.items(), key=lambda x: -x[1]):
    print(f"  {a:25s} â†’ {a:25s}  (x{w})")

Self-loops found: 0



## 2. Parallels (Bidirectional Flows)
Parallels are exactly the same as 2-node cycles â€” they indicate that in some cases a is followed by b and in others b is followed by a. This often hints at concurrency or interleaving.


In [6]:
edges = set(dfg_counter.keys())
two_node_loops = {}
seen = set()

for a, b in edges:
    if a == b or (b, a) in seen:
        continue
    if (b, a) in edges:
        w_ab = dfg_counter[(a, b)]
        w_ba = dfg_counter[(b, a)]
        two_node_loops[(a, b)] = (w_ab, w_ba)
        seen.add((a, b))


In [7]:
print("Parallel pairs (bidirectional edges):")
for (a, b), (w_ab, w_ba) in sorted(two_node_loops.items(), key=lambda x: -(x[1][0] + x[1][1])):
    ratio = w_ab / (w_ab + w_ba) if (w_ab + w_ba) else 0
    print(f"  {a:25s} â†” {b:25s}  ({ratio:.0%} aâ†’b, {1-ratio:.0%} bâ†’a, total: {w_ab + w_ba})")

Parallel pairs (bidirectional edges):


## 3. Longer Loops (3+ Nodes)
Use DFS to find cycles longer than 2 nodes in the DFG. This detects rework loops that span multiple activities.

In [8]:
def find_cycles(graph, min_length=3):
    adj = defaultdict(set)
    for a, b in graph:
        adj[a].add(b)

    cycles = []

    def dfs(node, start, visited, path):
        for neighbor in adj[node]:
            if neighbor == start and len(path) >= min_length:
                cycles.append(path + [neighbor])
            elif neighbor not in visited and neighbor > start:
                dfs(neighbor, start, visited | {neighbor}, path + [neighbor])

    for node in sorted(adj):
        dfs(node, node, {node}, [node])

    return cycles


cycles = find_cycles(set(dfg_counter.keys()))

print(f"Longer loops found (length â‰¥ 3): {len(cycles)}")
for cycle in cycles:
    print(f"  {' â†’ '.join(cycle)}")

Longer loops found (length â‰¥ 3): 0


## Summary
Aggregate all detected patterns into a single overview.

In [9]:
print("=" * 60)
print("DFG PATTERN DETECTION SUMMARY")
print("=" * 60)
print()

print(f"Total DFG edges:          {len(dfg_counter)}")
print(f"Self-loops (aâ†’a):         {len(self_loops)}")
print(f"2-node loops / parallels: {len(two_node_loops)}")
print(f"Longer cycles (â‰¥3 nodes): {len(cycles)}")
print()

all_loop_edges = set(self_loops.keys())
for (a, b) in two_node_loops:
    all_loop_edges.add((a, b))
    all_loop_edges.add((b, a))
loop_fraction = len(all_loop_edges) / len(dfg_counter) if dfg_counter else 0
print(f"Edges involved in any loop: {len(all_loop_edges)} / {len(dfg_counter)} ({loop_fraction:.1%})")

DFG PATTERN DETECTION SUMMARY

Total DFG edges:          31
Self-loops (aâ†’a):         0
2-node loops / parallels: 0
Longer cycles (â‰¥3 nodes): 0

Edges involved in any loop: 0 / 31 (0.0%)
